# 01. 데이터 생성

- 가상 H&B 플랫폼 '데일리글로우'의 사전 기간(2026-06-22 ~ 08-16) 데이터와 월별 KPI를 생성해 BigQuery `raw`에 적재함
- 생성 규칙은 `answer/generation_rules.md` 6장 (분석가는 보지 않는 정답 문서)
- 생성 방식: **세션 계획표 → 이벤트 펼치기** 2단계
  - 1단계: 세션마다 퍼널 도달·장바구니 금액·쇼핑 계속하기·결제 여부를 정답 문서 확률로 먼저 결정
  - 2단계: 계획표를 GA4 형식 이벤트 로그로 펼침
  - 정답 확률이 세션 단위로 정의돼 있어 정답 관리가 쉬움. 세션 계획표는 정답이 담긴 표라 BigQuery에 올리지 않음
- 전체 실행 시간: 약 3~5분 (이벤트 약 440만 행)

## 0. 설정

In [1]:
import sys, json, gzip, os, time            # sys: 경로 추가 / json: 이벤트를 JSON 문자열로 / gzip: 압축 저장 / os: 폴더 생성 / time: 실행 시간 측정
sys.path.append("..")                       # 노트북은 notebooks/ 안에 있으므로, 상위 폴더(리포지토리 최상단)의 config.py를 불러오기 위해 경로 추가
import numpy as np                          # 난수 생성과 배열 계산
import pandas as pd                         # 표(DataFrame) 처리
from config import SEED, PROJECT_ID, LOCATION, MAX_BYTES_BILLED   # 프로젝트 공통 설정 (시드, BigQuery 프로젝트 ID 등)

rng = np.random.default_rng(SEED)           # 모든 난수는 이 생성기 하나로 뽑음 → 같은 시드면 처음부터 끝까지 같은 데이터가 나옴
DATA_DIR = "../data"                        # 생성한 파일을 저장할 폴더 (리포지토리 최상단의 data/)
os.makedirs(DATA_DIR, exist_ok=True)        # 폴더가 없으면 만들고, 이미 있으면 그대로 둠

PRE_START = pd.Timestamp("2026-06-22")      # 사전 기간 시작일 (월요일)
PRE_END = pd.Timestamp("2026-08-16")        # 사전 기간 종료일 (일요일)
N_DAYS = (PRE_END - PRE_START).days + 1     # 사전 기간 일수 (56일 = 8주)
FREE_SHIP = 30000                           # 무료배송 기준 금액 (3만 원)
SHIP_FEE = 2500                             # 기준 미달 시 배송비
N_USERS = 100_000                           # 분석 대상 회원 수 (샘플)
T0 = time.time()                            # 실행 시간 측정 시작 시각


## 1라운드. 상품 마스터 (products)

- 카테고리별 기본 상품을 만들고, 색조는 호수·컬러 옵션마다 item_id를 따로 부여
- 미니·소용량 상품은 3천~8천 원 → 무료배송 금액을 채우기 좋은 상품
- `view_weight`: 탐색 화면 노출 가중치. 인기 상품 쏠림 + **미니 상품은 노출이 적게** (금액을 채울 상품을 찾기 어려운 이유)
- `is_color_option`: 색조 옵션 상품 표시 (백로그 이슈용)

In [2]:
# 카테고리별 생성 규칙: 기본 상품 수, 가격 범위(원), 미니·소용량 비중, 마진율 (정답 문서 6-3, 도메인 설정서 마진율)
CATEGORIES = {
    "스킨케어":     dict(n_base=150, price=(12000, 45000), mini=0.15, margin=0.40),
    "색조":         dict(n_base=20,  price=(9000, 35000),  mini=0.10, margin=0.45),   # 색조는 기본 상품 20개 × 옵션 5~15개
    "헤어·바디":    dict(n_base=90,  price=(6000, 25000),  mini=0.20, margin=0.30),
    "위생용품":     dict(n_base=90,  price=(3000, 15000),  mini=0.30, margin=0.20),
    "건강기능식품": dict(n_base=60,  price=(15000, 50000), mini=0.05, margin=0.35),
    "기타":         dict(n_base=30,  price=(3000, 20000),  mini=0.20, margin=0.30),
}
# 상품명에 쓸 단어 (검색어로도 재사용)
NAME_WORDS = {
    "스킨케어": ["수분 크림", "진정 토너", "비타민 세럼", "선크림", "클렌징 폼", "앰플", "아이크림", "시트 마스크"],
    "색조": ["쿠션 파운데이션", "리퀴드 파운데이션", "컨실러", "립 틴트", "립스틱", "아이섀도 팔레트", "블러셔", "아이브로우"],
    "헤어·바디": ["샴푸", "트리트먼트", "헤어 오일", "바디워시", "바디로션", "핸드크림"],
    "위생용품": ["생리대", "면봉", "화장솜", "치약", "칫솔", "렌즈 세정액", "물티슈", "데오드란트"],
    "건강기능식품": ["멀티비타민", "유산균", "오메가3", "콜라겐", "비타민C", "마그네슘"],
    "기타": ["메이크업 브러시", "퍼프", "헤어밴드", "파우치", "손톱깎이", "미니 거울"],
}
BASE_SHADES = ["13호", "17호", "19호", "21호", "22호", "23호", "25호"]            # 베이스 메이크업(파운데이션·컨실러) 옵션: 호수
COLOR_SHADES = ["01 코랄", "02 로즈", "03 레드", "04 베이지", "05 브라운", "06 핑크",  # 립·아이 등 옵션: 컬러
                "07 오렌지", "08 플럼", "09 누드", "10 모브", "11 체리", "12 피치",
                "13 버건디", "14 라벤더", "15 말린장미"]

rows = []                                   # 상품 1개(옵션 포함)씩 dict로 모아 마지막에 DataFrame으로 만듦
pid = 0                                     # item_id 번호
for cat, cfg in CATEGORIES.items():         # 카테고리마다
    for b in range(cfg["n_base"]):          # 기본 상품 수만큼 반복
        word = NAME_WORDS[cat][b % len(NAME_WORDS[cat])]          # 상품명 단어를 순서대로 돌려가며 사용
        is_mini = rng.random() < cfg["mini"]                      # 카테고리별 비중만큼 미니·소용량 상품으로 지정
        if is_mini:
            price = int(round(rng.uniform(3000, 8000), -2))       # 미니 상품은 3천~8천 원 (무료배송 금액을 채우기 좋은 가격대), 100원 단위 반올림
            base_name = f"{word} 미니 {b+1:03d}"
        else:
            price = int(round(rng.uniform(*cfg["price"]), -2))    # 일반 상품은 카테고리 가격 범위에서 균등하게 뽑음
            base_name = f"{word} {b+1:03d}"
        if cat == "색조":                                          # 색조만 옵션(호수·컬러)이 있음
            is_base_makeup = word in ("쿠션 파운데이션", "리퀴드 파운데이션", "컨실러")
            n_opt = int(rng.integers(5, 8)) if is_base_makeup else int(rng.integers(5, 16))   # 베이스는 호수 5~7개, 그 외 컬러 5~15개
            variants = BASE_SHADES[:n_opt] if is_base_makeup else COLOR_SHADES[:n_opt]
        else:
            n_opt, variants = 1, [None]                            # 옵션 없는 상품: 옵션 1개(None)로 처리
        for v in variants:                                         # 옵션마다 item_id를 따로 부여 (스키마: 옵션별로 별도 ID)
            pid += 1
            rows.append(dict(item_id=f"P{pid:04d}", item_name=base_name if v is None else f"{base_name} {v}",
                             item_category=cat, item_variant=v, option_count=n_opt, is_mini=bool(is_mini),
                             price=price, margin_rate=cfg["margin"], base_name=base_name))
products = pd.DataFrame(rows)
# 탐색 노출 가중치: 파레토 분포로 인기 상품 쏠림을 만들고, 미니·소용량은 노출을 30%로 줄임
# → 고객이 금액을 채울 3천~8천 원 상품을 탐색 화면에서 찾기 어려운 이유 (정답 문서 6-3)
w = rng.pareto(1.5, len(products)) + 1
w = np.where(products["is_mini"], w * 0.3, w)
products["view_weight"] = w / w.sum()                                              # 합이 1이 되도록 정규화 (확률로 사용)
products["is_color_option"] = (products["item_category"] == "색조") & (products["option_count"] >= 5)   # 백로그 이슈 대상 상품 표시
print("products", len(products), products["item_category"].value_counts().to_dict())


products 587 {'색조': 167, '스킨케어': 150, '헤어·바디': 90, '위생용품': 90, '건강기능식품': 60, '기타': 30}


## 2라운드. 회원 (users)

- 세그먼트: 온라인 이용 55% / 매장 전용 15% / 신규 30% (사전 기간 시작일 기준)
- user_id는 무작위로 섞어서 ID 순서로 세그먼트가 드러나지 않게 함
- 첫 구매일은 기간 시작 전 기준으로 먼저 만들고, 5라운드에서 기간 중 첫 온라인 구매가 생기면 갱신 (데이터 추출 시점 스냅샷)
- 플랫폼·기기·유입 경로는 회원 단위로 고정 (신규 가입자의 80%는 광고 유입)

In [3]:
SEG_SHARE = {"online": 0.55, "store": 0.15, "new": 0.30}                   # 세그먼트 비중 (온라인 이용 / 매장 전용 / 신규)
seg = rng.choice(list(SEG_SHARE), size=N_USERS, p=list(SEG_SHARE.values()))   # 회원마다 세그먼트를 확률로 배정
users = pd.DataFrame({"segment": seg})                                      # segment는 정답 정보라 BigQuery에는 올리지 않음
users["user_id"] = [f"U{n:06d}" for n in rng.permutation(N_USERS) + 1]      # 번호를 섞어서 부여 → ID 순서로 세그먼트가 드러나지 않음
# user_pseudo_id: GA4의 기기 식별자 형식(숫자.숫자)을 흉내 냄
users["user_pseudo_id"] = [f"{a:010d}.{b:010d}" for a, b in zip(rng.integers(1e9, 1e10, N_USERS), rng.integers(1.7e9, 1.78e9, N_USERS))]

def rand_dates(start, end, n):
    """start ~ end 사이 날짜를 n개 균등하게 뽑음"""
    days = (pd.Timestamp(end) - pd.Timestamp(start)).days + 1               # 기간 일수
    return pd.Timestamp(start) + pd.to_timedelta(rng.integers(0, days, n), unit="D")

n = N_USERS
is_on, is_st, is_new = (seg == "online"), (seg == "store"), (seg == "new")  # 세그먼트별 True/False 배열
new_in_period = is_new & (rng.random(n) < 0.60)                              # 신규 중 60%는 사전 기간 중 가입 (마케팅 유입)
signup = rand_dates("2021-01-01", "2026-06-21", n)                           # 기본: 기간 시작 전 가입
signup = signup.where(~new_in_period, rand_dates(PRE_START, PRE_END, n))     # 기간 중 가입 신규 → 가입일을 기간 안으로
signup = signup.where(~(is_new & ~new_in_period), rand_dates("2025-06-01", "2026-06-21", n))   # 나머지 신규 → 최근 1년 내 가입 후 미구매
users["signup_date"] = signup
users["new_in_period"] = new_in_period                                       # 기간 중 가입 여부 (세션 생성에 사용, BigQuery에는 안 올림)
ch_online_p = np.select([is_on, is_st, is_new], [0.70, 0.10, 0.85])          # 세그먼트별 '온라인 가입' 확률 (매장 전용은 대부분 매장 가입)
users["signup_channel"] = np.where(rng.random(n) < ch_online_p, "online", "store")

def date_between(a, b):
    """a와 b 사이(양 끝 포함) 날짜를 행마다 무작위로 뽑음 (a > b이면 a)"""
    a = pd.Series(a).reset_index(drop=True); b = pd.Series(b).reset_index(drop=True)
    span = (b - a).dt.days.clip(lower=0)                                      # 두 날짜 사이 일수 (음수면 0)
    return a + pd.to_timedelta((rng.random(len(a)) * (span + 1)).astype(int), unit="D")

first_online = pd.Series(pd.NaT, index=users.index, dtype="datetime64[ns]")  # 첫 온라인 구매일 (기본: 없음)
first_store = pd.Series(pd.NaT, index=users.index, dtype="datetime64[ns]")   # 첫 매장 구매일 (기본: 없음)
# 온라인 이용 고객: 가입일 ~ 기간 시작 전날 사이에 첫 온라인 구매
first_online[is_on] = date_between(users.loc[is_on, "signup_date"], [PRE_START - pd.Timedelta(days=1)] * is_on.sum()).values
has_store = is_on & (rng.random(n) < 0.60)                                    # 온라인 이용 고객 중 60%는 매장 구매 이력도 있음
first_store[has_store] = date_between([pd.Timestamp("2019-01-01")] * has_store.sum(), users.loc[has_store, "signup_date"]).values
# 매장 전용 고객: 매장 구매 이력만 있음 (온라인은 없음)
first_store[is_st] = date_between([pd.Timestamp("2019-01-01")] * is_st.sum(), users.loc[is_st, "signup_date"]).values
users["first_online_purchase_date"] = first_online                            # 5라운드에서 기간 중 첫 구매가 생기면 갱신
users["first_store_purchase_date"] = first_store

grades = ["pearl", "gold", "diamond", "black_diamond"]                        # 멤버십 등급 (낮음 → 높음)
grade_p = {"online": [0.40, 0.35, 0.20, 0.05], "store": [0.50, 0.35, 0.13, 0.02], "new": [1, 0, 0, 0]}   # 신규는 구매 이력이 없어 모두 최하 등급
users["membership_grade"] = [grades[rng.choice(4, p=grade_p[s])] for s in seg]

plat = np.where(rng.random(n) < 0.80, "APP", "WEB")                           # 앱 80% / 웹 20% (도메인 설정서)
os_app = np.where(rng.random(n) < 0.55, "iOS", "Android")                     # 앱 사용자 OS
os_web = rng.choice(["Windows", "Macintosh", "iOS", "Android"], n, p=[0.35, 0.15, 0.30, 0.20])   # 웹 사용자 OS (모바일 웹 포함)
users["platform"] = plat
users["operating_system"] = np.where(plat == "APP", os_app, os_web)

# 최초 유입 경로: 기간 중 가입 신규의 80%는 광고 유입, 나머지는 직접 방문 또는 네이버 검색
paid = new_in_period & (rng.random(n) < 0.80)
src = rng.choice(["google", "meta", "naver"], n, p=[0.40, 0.35, 0.25])
users["ts_source"] = np.where(paid, src, np.where(rng.random(n) < 0.6, "(direct)", "naver"))
users["ts_medium"] = np.where(paid, np.where(src == "meta", "paid_social", "cpc"), np.where(users["ts_source"] == "(direct)", "(none)", "organic"))
users["ts_name"] = np.where(paid, "2026_summer_signup", "(direct)")           # 광고 캠페인명
print("users", users["segment"].value_counts().to_dict(), round(time.time() - T0, 1), "s")


users {'online': 54809, 'new': 30076, 'store': 15115} 0.5 s


## 3라운드. 세션 계획표 (비공개)

- 회원별 세션 수: 포아송 분포 (평균 4.0 / 2.0 / 3.0)
- 세션 날짜: 가입일 이후 ~ 기간 종료, 주말 비중을 높임 / 시각: 저녁 20~23시 비중 높음, 0~2시도 일부 (KST·UTC 변환 확인용)
- 퍼널 확률(상세 조회, 담기, 장바구니 진입)과 목표 장바구니 금액(로그정규분포)을 세션마다 결정

In [4]:
# 세그먼트별 행동 규칙 (정답 문서 6-2, 6-4, 6-5, 6-9)
#   sess: 8주간 평균 세션 수 / detail: 상품 상세 조회 확률 / add: 장바구니 담기 확률 / cart: 장바구니 진입 확률
#   median: 장바구니 금액 중앙값 / ret: 쇼핑 계속하기 후 돌아와 금액을 채울 확률 / pay: 주문서 → 결제 완료 확률
#   late_login: 로그인 전 둘러보는 세션 비율 (품질 이슈: user_id 누락)
SEG = {
    "online": dict(sess=4.0, detail=0.75, add=0.45, cart=0.90, median=42000, ret=0.65, pay=0.93, late_login=0.15),
    "store":  dict(sess=2.0, detail=0.70, add=0.35, cart=0.85, median=24000, ret=0.35, pay=0.90, late_login=0.15),
    "new":    dict(sess=3.0, detail=0.70, add=0.35, cart=0.85, median=22000, ret=0.30, pay=0.90, late_login=0.30),
}
# 부족 금액 구간: 0=기준 이상 / 1=1~2,999원 / 2=3,000~7,999원 / 3=8,000~14,999원 / 4=15,000원 이상
CLICK = np.array([0.0, 0.40, 0.55, 0.35, 0.15])    # 구간별 '쇼핑 계속하기' 클릭 확률 (3천~8천 원 구간이 가장 높음)
DIRECT = np.array([0.72, 0.55, 0.45, 0.55, 0.60])  # 클릭 안 한 경우 바로 주문서로 가는 확률 (15,000원 이상 부족하면 배송비 내고 구매)

def gap_bucket(gap):
    """부족 금액 → 구간 번호(0~4)"""
    return np.select([gap <= 0, gap < 3000, gap < 8000, gap < 15000], [0, 1, 2, 3], 4)

sess_mean = users["segment"].map({k: v["sess"] for k, v in SEG.items()}).to_numpy()   # 회원별 평균 세션 수
n_sess = np.clip(rng.poisson(sess_mean), 1, None)                     # 포아송 분포로 세션 수를 뽑고, 최소 1회 (방문 회원 샘플이므로)
s = pd.DataFrame({"u": np.repeat(np.arange(N_USERS), n_sess)})       # 세션 계획표: 1행 = 1세션, u = 회원 행 번호
s["segment"] = users["segment"].to_numpy()[s["u"]]
S = len(s)                                                            # 전체 세션 수
P = {k: s["segment"].map({g: v[k] for g, v in SEG.items()}).to_numpy() for k in SEG["online"]}   # 세션별 확률 배열 (아래에서 정렬 후 다시 계산)

# 세션 날짜: 가입일(또는 기간 시작일) ~ 기간 종료일 사이에서 뽑음
start = np.maximum(users["signup_date"].to_numpy()[s["u"]], np.datetime64(PRE_START))   # 기간 중 가입자는 가입일 이후에만 방문
span = ((np.datetime64(PRE_END) - start) / np.timedelta64(1, "D")).astype(int) + 1      # 방문 가능한 일수
day_off = (rng.random(S) * span).astype(int)                          # 시작일로부터 며칠 뒤인지
dates = start + day_off.astype("timedelta64[D]")
dow = pd.DatetimeIndex(dates).dayofweek                               # 요일 (0=월 ~ 6=일)
redraw = (dow < 5) & (rng.random(S) < 0.17)                           # 평일 세션 중 17%를 다시 뽑아 주말 비중을 살짝 높임
day_off2 = (rng.random(S) * span).astype(int)
dates = np.where(redraw, start + day_off2.astype("timedelta64[D]"), dates)
# 시간대 가중치 (0시 ~ 23시): 저녁 20~23시가 가장 높고 새벽 3~5시가 가장 낮음, 0~2시에도 일부 발생 (KST·UTC 변환 확인용)
HOUR_W = np.array([3, 2, 1, .5, .3, .3, .5, 1.5, 3, 4, 4, 4, 5, 4, 4, 4, 4, 5, 6, 7, 9, 10, 9, 6], float)
hour = rng.choice(24, S, p=HOUR_W / HOUR_W.sum())
sec = rng.integers(0, 3600, S)                                        # 해당 시간 안의 초
s["start_kst"] = pd.to_datetime(dates) + pd.to_timedelta(hour, unit="h") + pd.to_timedelta(sec, unit="s")   # 세션 시작 시각 (KST)
s = s.sort_values(["u", "start_kst"]).reset_index(drop=True)          # 회원별 시간순 정렬
s["session_number"] = s.groupby("u").cumcount() + 1                   # 회원의 몇 번째 세션인지 (GA4 ga_session_number)
s["segment"] = users["segment"].to_numpy()[s["u"]]                    # 정렬 후 세그먼트 다시 매칭
P = {k: s["segment"].map({g: v[k] for g, v in SEG.items()}).to_numpy() for k in SEG["online"]}   # 정렬된 순서로 확률 배열 다시 계산
s["is_signup_session"] = (s["session_number"] == 1) & users["new_in_period"].to_numpy()[s["u"]]   # 기간 중 가입자의 첫 세션 = 가입 세션
s["late_login"] = rng.random(S) < P["late_login"]                     # 로그인 전에 둘러보는 세션 (품질 이슈)

s["detail"] = rng.random(S) < P["detail"]                             # 상품 상세를 봤는지
# 장바구니 담기 여부는 다음 셀에서 상품을 정한 뒤 확정 (색조 옵션 상품은 담기 확률이 낮아서)
s["add_draw"] = rng.random(S)                                         # 담기 판정용 난수를 미리 뽑아둠
s["cart_draw"] = rng.random(S) < P["cart"]                            # 담은 뒤 장바구니 화면에 들어갔는지
s["target_amount"] = np.round(np.exp(rng.normal(np.log(P["median"]), 0.45)), -2)   # 목표 장바구니 금액 (로그정규분포, 100원 단위)
print("sessions", S, round(time.time() - T0, 1), "s")


sessions 344287 0.9 s


### 3라운드-2. 세션별 상품 구성과 장바구니 행동

- 목표 금액을 넘지 않게 상품을 하나씩 담아 실제 장바구니 금액을 만듦
- 부족 금액 구간별로 '쇼핑 계속하기' 클릭 → 복귀(금액 채움) 또는 이탈을 결정 (정답 문서 6-5, U자형)
- 색조 옵션 상품은 담기 확률을 낮춤 (백로그 이슈)

In [5]:
pr = products
price = pr["price"].to_numpy()                                        # 상품 가격 배열 (상품 행 번호로 바로 조회)
is_co = pr["is_color_option"].to_numpy()                              # 색조 옵션 상품 여부 배열
# 상품을 하나씩 rng.choice로 뽑으면 느리므로, 노출 가중치대로 충분히 많이 미리 뽑아두고 순서대로 꺼내 씀
POOL = rng.choice(len(pr), size=S * 12, p=pr["view_weight"].to_numpy())
ptr = 0                                                               # POOL에서 다음에 꺼낼 위치
def nxt():
    """미리 뽑아둔 상품 풀에서 다음 상품 1개를 꺼냄"""
    global ptr
    v = POOL[ptr]; ptr += 1
    return v

cheap_idx = np.where(price <= 8000)[0]                                # 8천 원 이하 상품 (예외 처리용)
viewed, carted, added = [], [], []                                    # 세션별: 상세 조회한 상품 / 장바구니에 담은 상품 / 담기 여부
detail = s["detail"].to_numpy(); add_draw = s["add_draw"].to_numpy(); target = s["target_amount"].to_numpy()
add_p = P["add"]
for i in range(S):
    if not detail[i]:                                                 # 상세를 안 본 세션: 담은 상품 없음
        viewed.append([]); carted.append([]); added.append(False); continue
    first = nxt()                                                     # 처음 본 상품
    factor = 0.70 if is_co[first] else 1.10                           # 색조 옵션 상품이면 담기 확률을 낮춤 (호수·컬러 고르기 어려움, 백로그 이슈)
    if add_draw[i] < add_p[i] * factor:                               # 담기 성공
        items, rem, tries = [], target[i], 0                          # rem: 목표 금액까지 남은 돈
        if price[first] <= rem:
            items.append(first); rem -= price[first]
        while rem >= 3000 and len(items) < 8 and tries < 12:          # 목표 금액을 넘지 않게 상품을 더 담음 (최대 8개)
            c = nxt()
            if price[c] <= rem:
                items.append(c); rem -= price[c]
            else:
                tries += 1                                            # 너무 비싼 상품이 12번 나오면 그만 담음
        if not items:                                                 # 하나도 못 담았으면 저가 상품 1개를 담음
            items = [int(rng.choice(cheap_idx))]
        extra = [nxt() for _ in range(rng.integers(0, 3))]            # 담지 않고 구경만 한 상품 0~2개
        viewed.append(items + extra); carted.append(items); added.append(True)
    else:                                                             # 담기 실패: 구경만 함
        viewed.append([first] + [nxt() for _ in range(rng.integers(0, 3))]); carted.append([]); added.append(False)
s["viewed"] = viewed
s["carted"] = carted
s["added"] = added
s["cart"] = s["added"] & s["cart_draw"]                               # 담고 + 장바구니 화면까지 간 세션
s["cart_amount"] = [int(price[c].sum()) if len(c) else 0 for c in carted]   # 실제 장바구니 금액 (담은 상품 가격 합)
s["gap"] = np.where(s["cart"], np.clip(FREE_SHIP - s["cart_amount"], 0, None), 0)   # 무료배송까지 부족한 금액 (기준 이상이면 0)
b = gap_bucket(s["gap"].to_numpy())                                   # 부족 금액 구간
cart = s["cart"].to_numpy(); gap = s["gap"].to_numpy()
s["clicked"] = cart & (gap > 0) & (rng.random(S) < CLICK[b])          # 기준 미달 장바구니에서 '쇼핑 계속하기' 클릭
s["returned"] = s["clicked"] & (rng.random(S) < P["ret"])             # 클릭 후 상품을 찾아 장바구니로 돌아옴 (세그먼트별 확률)
# 주문서 진입: 클릭한 경우 → 돌아온 사람의 85%, 클릭 안 한 경우 → 구간별 확률
s["checkout"] = np.where(s["clicked"], s["returned"] & (rng.random(S) < 0.85), cart & (rng.random(S) < DIRECT[b]))
s["purchase"] = s["checkout"] & (rng.random(S) < P["pay"])            # 주문서 → 결제 완료
s["gap_bucket"] = b

# 금액을 채운 상품 (부족 금액 ~ 부족 금액 + 5,000원), 찾다가 둘러본 상품 (부족 금액 근처 가격)
fill_item, hunt_items = [], []
for i in range(S):
    if s.at[i, "clicked"]:
        g = gap[i]
        near = np.where((price >= g * 0.7) & (price <= g + 5000))[0]  # 부족 금액 근처 가격대 상품
        if len(near) == 0:
            near = cheap_idx
        hunt = list(rng.choice(near, size=int(rng.integers(2, 6))))   # 2~5개를 둘러봄 → "이탈 직전 조회 상품 가격이 부족 금액 근처" 신호
        hunt_items.append(hunt)
        if s.at[i, "returned"]:
            ok = np.where((price >= g) & (price <= g + 5000))[0]      # 부족 금액을 채우는 상품
            fill_item.append(int(rng.choice(ok if len(ok) else near)))
        else:
            fill_item.append(None)                                    # 못 찾고 이탈
    else:
        hunt_items.append([]); fill_item.append(None)
s["hunt_items"] = hunt_items
s["fill_item"] = fill_item
s["final_amount"] = s["cart_amount"] + [price[f] if f is not None else 0 for f in fill_item]   # 최종 장바구니 금액
s["shipping"] = np.where(s["final_amount"] >= FREE_SHIP, 0, SHIP_FEE)                          # 배송비
print("plan done", round(time.time() - T0, 1), "s")


plan done 3.9 s


## 4라운드. 이벤트 로그 (events, GA4 형식)

- 세션 계획표 1행을 이벤트 여러 행으로 펼침 → `data/events.jsonl.gz` (줄마다 JSON 1개)
- 메모리를 아끼기 위해 DataFrame으로 모으지 않고 파일에 바로 씀
- `event_date`는 KST 날짜, `event_timestamp`는 UTC 마이크로초 (03 설계서 기준)
- **품질 이슈 2개를 여기서 함께 심음**
  1. 로그인 전 이벤트의 user_id 누락 (`late_login` 세션)
  2. purchase 이벤트 중복 전송 0.8% (같은 transaction_id, 1초 이내 차이)

In [6]:
# 자주 쓰는 컬럼을 numpy 배열로 꺼내둠 (행 번호로 빠르게 조회)
p_id = pr["item_id"].to_numpy(); p_opt = pr["option_count"].to_numpy(); p_var = pr["item_variant"].to_numpy()
p_cat = pr["item_category"].to_numpy(); p_name = pr["item_name"].to_numpy()
U = users
u_uid = U["user_id"].to_numpy(); u_pid = U["user_pseudo_id"].to_numpy(); u_plat = U["platform"].to_numpy()
u_os = U["operating_system"].to_numpy(); u_grade = U["membership_grade"].to_numpy(); u_ch = U["signup_channel"].to_numpy()
u_src = U["ts_source"].to_numpy(); u_med = U["ts_medium"].to_numpy(); u_name = U["ts_name"].to_numpy()
METHODS = ["kakao", "naver", "email"]   # 가입·로그인 방식
DUP_RATE = 0.008                        # 품질 이슈: purchase 이벤트 중복 전송 비율 (0.8%)

def build_session(r):
    """세션 계획표 1행 → 이벤트 목록 [(경과 초, 이벤트명, 파라미터, 상품 목록, 화면, 직전 화면)], 주문 정보"""
    E = []                                            # 이 세션의 이벤트 목록
    st = {"t": 0.0, "screen": "home", "prev": ""}     # 현재 경과 시간(초), 현재 화면, 직전 화면
    def add(name, params=None, items=None, gap=(3, 40)):
        """이벤트 1개 추가: gap 범위(초)만큼 시간이 흐른 뒤 발생"""
        st["t"] += rng.uniform(*gap)
        E.append((st["t"], name, params or {}, items, st["screen"], st["prev"]))
    def go(screen):
        """화면 이동: 이후 이벤트의 screen_name / previous_screen_name이 바뀜"""
        st["prev"], st["screen"] = st["screen"], screen
    def login():
        add("login", {"method": METHODS[rng.choice(3, p=[.5, .3, .2])]}, None, (5, 20))

    E.append((0.0, "session_start", {}, None, "home", ""))   # 모든 세션은 홈 화면 session_start로 시작
    logged_in = not r.late_login                               # late_login 세션은 로그인 전 상태로 시작
    if r.is_signup_session:                                    # 가입 세션: 가입 → 로그인
        add("sign_up", {"method": METHODS[rng.choice(3, p=[.5, .3, .2])]})
        login(); logged_in = True

    carted = set(r.carted)                                     # 이 세션에서 담을 상품 (담으면 하나씩 지움)
    if not r.viewed and rng.random() < 0.5:                    # 상세를 안 본 세션 중 절반은 목록만 보고 나감
        go("category"); add("view_item_list", {"item_list_name": "category"}, [int(nxt())])
    for it in r.viewed:                                        # 상세를 본 상품마다
        if rng.random() < 0.4:                                 # 40%는 검색으로 찾음
            go("search")
            add("search", {"search_term": NAME_WORDS[p_cat[it]][rng.integers(len(NAME_WORDS[p_cat[it]]))], "result_count": int(rng.integers(5, 120))})
            lst = "search"
        else:                                                  # 60%는 카테고리로 찾음
            go("category"); lst = "category"
        add("view_item_list", {"item_list_name": lst}, [it])               # 목록 노출
        add("select_item", {"item_list_name": lst}, [it], (2, 10))         # 목록에서 상품 선택
        go("product"); add("view_item", {"value": int(price[it])}, [it])   # 상품 상세 진입
        if is_co[it]:                                 # 색조 옵션 상품: 옵션 선택창·리뷰 탐색이 많음 (백로그 이슈, 정답 문서 6-6)
            n_open = 2 + int(rng.integers(0, 3)) if rng.random() < 0.45 else 1   # 45%는 옵션 선택창을 2~4번 열고 닫음
            for _ in range(n_open):
                add("open_option_selector", {"item_id": p_id[it], "option_count": int(p_opt[it])}, None, (3, 20))
            if rng.random() < 0.60:                                              # 60%는 리뷰 탭 진입
                add("view_review", {"item_id": p_id[it]}, None, (5, 60))
                if rng.random() < 0.25 / 0.60:                                   # 전체의 25%는 리뷰에서 호수 키워드 검색
                    add("search_review", {"item_id": p_id[it], "search_term": ["21호", "23호", "17호", "19호", "건성", "지성"][rng.integers(6)]}, None, (5, 30))
            add("select_option", {"item_id": p_id[it], "item_variant": p_var[it]}, None, (3, 20))
        elif rng.random() < 0.30:                                                # 그 외 상품은 30%만 리뷰 탭 진입
            add("view_review", {"item_id": p_id[it]}, None, (5, 60))
            if rng.random() < 0.02 / 0.30:                                       # 전체의 2%만 리뷰 검색
                add("search_review", {"item_id": p_id[it], "search_term": ["향", "용량", "후기"][rng.integers(3)]}, None, (5, 30))
        if it in carted:                              # 담을 상품이면 장바구니 담기
            if not logged_in:                         # 장바구니 담기는 로그인 필요 → 이 시점에 로그인
                login(); logged_in = True
            add("add_to_cart", {"value": int(price[it]), "add_source": "product"}, [it], (3, 15))
            carted.discard(it)
    if not logged_in:                                 # 끝까지 로그인 안 했으면 마지막에 로그인 (모든 세션에 로그인이 있어 세션 단위로 user_id 복원 가능)
        login(); logged_in = True

    order = None
    if r.cart:                                        # 장바구니 화면까지 간 세션
        amt, g = int(r.cart_amount), int(r.gap)       # 장바구니 금액, 부족 금액
        items = [int(x) for x in r.carted]
        sources = ["product"] * len(items)            # 상품별 담긴 경로 (order_items.add_source)
        go("cart"); add("view_cart", {"value": amt, "free_shipping_remaining": g, "item_count": len(items)}, items)
        if r.clicked:                                 # '쇼핑 계속하기' 클릭
            add("click_continue_shopping", {"value": amt, "free_shipping_remaining": g}, None, (3, 15))
            go("category"); add("view_item_list", {"item_list_name": "category"}, [int(h) for h in r.hunt_items])
            for h in r.hunt_items:                    # 부족 금액 근처 가격의 상품을 찾아다님
                add("select_item", {"item_list_name": "category"}, [int(h)], (2, 10))
                go("product"); add("view_item", {"value": int(price[h])}, [int(h)])
                go("category")
            if r.returned:                            # 금액을 채울 상품을 찾음 → 담고 장바구니로 복귀
                f = int(r.fill_item)
                go("product")
                src_f = "list" if rng.random() < 0.3 else "product"     # 30%는 목록에서 바로 담기
                add("add_to_cart", {"value": int(price[f]), "add_source": src_f}, [f], (3, 15))
                items, amt = items + [f], amt + int(price[f])
                sources = sources + [src_f]
                go("cart"); add("view_cart", {"value": amt, "free_shipping_remaining": max(FREE_SHIP - amt, 0), "item_count": len(items)}, items)
            # 돌아오지 않은 경우: 여기서 세션이 끝남 (이탈)
        if r.checkout:                                # 주문서 진입
            ship = 0 if amt >= FREE_SHIP else SHIP_FEE
            add("begin_checkout", {"value": amt, "shipping": ship}, items)
            go("checkout")
            reach = 3 if r.purchase else int(rng.choice(3, p=[.4, .35, .25]))   # 결제 못 한 경우 주문서 몇 단계까지 갔는지
            if reach >= 1:
                add("add_shipping_info", {"shipping_saved": "Y" if r.segment == "online" else "N"}, None, (5, 40))   # 온라인 이용 고객만 저장된 배송지 사용
            if reach >= 2:
                ptype = ["card", "easy_pay", "etc"][rng.choice(3, p=[.45, .50, .05])]
                add("add_payment_info", {"payment_type": ptype}, None, (5, 30))
            if r.purchase:                            # 결제 완료 → 주문 정보 반환
                add("purchase", {"value": amt, "shipping": ship}, items, (3, 20))
                order = dict(item_amount=amt, shipping_fee=ship, payment_type=ptype, items=items, sources=sources, offset=E[-1][0])
    return E, order

def param(k, v):
    """GA4 event_params 형식으로 변환: 값 타입에 따라 string_value / int_value / double_value 중 하나만 채움"""
    if isinstance(v, (bool, np.bool_)):
        v = "Y" if v else "N"
    if isinstance(v, (int, np.integer)):
        return {"key": k, "value": {"int_value": int(v)}}
    if isinstance(v, float):
        return {"key": k, "value": {"double_value": v}}
    return {"key": k, "value": {"string_value": str(v)}}

def item_obj(ix):
    """상품 행 번호 → GA4 items 배열의 원소"""
    v = p_var[ix]
    return {"item_id": p_id[ix], "item_name": p_name[ix], "item_category": p_cat[ix],
            # 옵션 없는 상품은 pandas에서 NaN으로 저장됨 → JSON에 NaN이 쓰이면 BigQuery가 읽지 못하므로 null(None)로 바꿈
            "item_variant": v if isinstance(v, str) else None,
            "price": float(price[ix]), "quantity": 1}

# 시각 계산을 빠르게 하기 위해 세션 시작 시각을 "KST 기준 epoch 초"(숫자)로 바꿔둠
start_kst_sec = (s["start_kst"] - pd.Timestamp("1970-01-01")).dt.total_seconds().to_numpy()
start_kst_ts = s["start_kst"].to_numpy()
KST_SEC = 9 * 3600                                                # KST = UTC + 9시간
DATE_STR = {}                                                     # 일 번호 → "YYYYMMDD" 문자열 캐시 (같은 날짜를 매번 변환하지 않도록)
def kst_date(sec):
    """KST 기준 epoch 초 → 'YYYYMMDD' (GA4 event_date 형식)"""
    d = int(sec // 86400)
    if d not in DATE_STR:
        DATE_STR[d] = (pd.Timestamp("1970-01-01") + pd.Timedelta(days=d)).strftime("%Y%m%d")
    return DATE_STR[d]

orders_tmp, n_events, n_dup = [], 0, 0                            # 주문 정보 모음, 이벤트 수, 중복 이벤트 수
# 이벤트가 400만 행이 넘어 DataFrame으로 모으면 메모리를 많이 쓰므로, 한 줄씩 압축 파일에 바로 씀 (줄마다 JSON 1개 = NDJSON)
with gzip.open(f"{DATA_DIR}/events.jsonl.gz", "wt", encoding="utf-8") as fout:
    for r in s.itertuples():                                      # 세션 계획표를 한 행씩
        u = r.u                                                   # 회원 행 번호
        E, order = build_session(r)
        s0 = start_kst_sec[r.Index]                               # 세션 시작 (KST epoch 초)
        ga_session_id = int(s0 - KST_SEC)                         # GA4처럼 세션 시작 시각(UTC epoch 초)을 세션 ID로 사용
        # late_login 세션: 로그인(또는 가입) 시점 이전 이벤트는 user_id를 비움
        login_t = next((t for t, n_, *_ in E if n_ in ("login", "sign_up")), 0.0) if r.late_login else -1.0
        tid = None
        if order:                                                 # 결제한 세션이면 주문 번호 발급
            tid = f"T{len(orders_tmp) + 1:07d}"
            order.update(sidx=r.Index, transaction_id=tid,
                         order_datetime=pd.Timestamp(start_kst_ts[r.Index]) + pd.Timedelta(seconds=order["offset"]))
            orders_tmp.append(order)
        for t, name, params, items, scr, prev in E:               # 이벤트를 GA4 형식 1행으로 변환
            kst_sec = s0 + t                                      # 이벤트 발생 시각 (KST epoch 초)
            ts_utc_us = int(round((kst_sec - KST_SEC) * 1_000_000))   # event_timestamp: UTC 기준 마이크로초 (03 설계서)
            ep = [param("ga_session_id", ga_session_id), param("ga_session_number", int(r.session_number)),
                  param("screen_name", scr), param("previous_screen_name", prev)]    # 모든 이벤트에 붙는 공통 파라미터
            if name == "purchase":
                params = {"transaction_id": tid, **params}       # purchase에는 주문 번호 추가
            ep += [param(k, v) for k, v in params.items()]       # 이벤트별 파라미터
            row = {
                "event_date": kst_date(kst_sec),                  # event_date: KST 기준 날짜 문자열
                "event_timestamp": ts_utc_us,
                "event_name": name,
                "event_params": ep,
                # 품질 이슈 1: 로그인 전 이벤트는 user_id 없음 (None → JSON null)
                "user_id": u_uid[u] if (not r.late_login or t >= login_t) else None,
                "user_pseudo_id": u_pid[u],
                "platform": u_plat[u],
                "device": {"category": "mobile" if u_plat[u] == "APP" or u_os[u] in ("iOS", "Android") else "desktop",
                           "operating_system": u_os[u]},
                "traffic_source": {"name": u_name[u], "medium": u_med[u], "source": u_src[u]},   # 최초 유입 경로 (회원 단위 고정)
                "user_properties": [param("membership_grade", u_grade[u]), param("signup_channel", u_ch[u])],
                "items": [item_obj(ix) for ix in items] if items else [],   # 상품 이벤트가 아니면 빈 배열
                "ecommerce": ({"transaction_id": tid, "purchase_revenue": float(params["value"] + params["shipping"]),
                               "shipping_value": float(params["shipping"]), "total_item_quantity": len(items)}
                              if name == "purchase" else None),             # purchase에만 주문 금액 정보
            }
            line = json.dumps(row, ensure_ascii=False)            # 한글을 그대로 저장 (ensure_ascii=False)
            fout.write(line + "\n"); n_events += 1
            if name == "purchase" and rng.random() < DUP_RATE:    # 품질 이슈 2: purchase 이벤트 중복 전송 (0.8%)
                row["event_timestamp"] += int(rng.integers(1, 1_000_000))   # 1초 이내 차이로 한 번 더 기록
                fout.write(json.dumps(row, ensure_ascii=False) + "\n"); n_events += 1; n_dup += 1
print("events", n_events, "dup", n_dup, "orders", len(orders_tmp), round(time.time() - T0, 1), "s")


events 4390472 dup 412 orders 51523 76.8 s


## 5라운드. 주문·주문 상세·반품

- 결제한 세션에서 주문을 만들고, 이벤트의 transaction_id와 order_id를 맞춤
- 반품: 색조 옵션 상품 옵션 불만 6% / 그 외 1% + 기타 사유 2%, 데이터 추출일(8/17) 이후 접수 건은 제외
- users의 첫 온라인 구매일 갱신

In [7]:
# orders: 결제한 세션 1개 = 주문 1건 (order_id = 이벤트의 transaction_id)
orders = pd.DataFrame({
    "order_id": [o["transaction_id"] for o in orders_tmp],
    "user_id": [u_uid[s.at[o["sidx"], "u"]] for o in orders_tmp],
    "order_datetime": [o["order_datetime"].floor("s") for o in orders_tmp],   # 주문 일시 (KST, 초 단위로 자름)
    "platform": [u_plat[s.at[o["sidx"], "u"]] for o in orders_tmp],
    "item_amount": [o["item_amount"] for o in orders_tmp],
    "shipping_fee": [o["shipping_fee"] for o in orders_tmp],
    "payment_type": [o["payment_type"] for o in orders_tmp],
})
orders["total_amount"] = orders["item_amount"] + orders["shipping_fee"]  # 결제 금액 = 상품 금액 + 배송비
orders["order_status"] = "completed"                                     # 취소 주문은 생성하지 않음 (정답 문서 6-7)
orders = orders[["order_id", "user_id", "order_datetime", "platform", "item_amount", "shipping_fee",
                 "total_amount", "payment_type", "order_status"]]        # 스키마 컬럼 순서로 정리

# order_items: 주문 1건 안의 상품들을 한 행씩 펼침
oi = pd.DataFrame([(o["transaction_id"], p_id[ix], int(price[ix]), src)
                   for o in orders_tmp for ix, src in zip(o["items"], o["sources"])],
                  columns=["order_id", "item_id", "price", "add_source"])
order_items = (oi.groupby(["order_id", "item_id", "price", "add_source"], as_index=False)
                 .size().rename(columns={"size": "quantity"}))           # 같은 상품을 여러 번 담았으면 수량으로 합침
order_items = order_items[["order_id", "item_id", "price", "quantity", "add_source"]]

# returns: 색조 옵션 상품은 옵션 불만 6%, 그 외 1% + 기타 사유(단순 변심·파손·오배송) 2% (정답 문서 6-6, 6-7)
oi2 = order_items.merge(products[["item_id", "is_color_option"]], on="item_id").merge(orders[["order_id", "order_datetime"]], on="order_id")
p_opt_ret = np.where(oi2["is_color_option"], 0.06, 0.01)                # 상품별 옵션 불만 반품 확률
r_draw = rng.random(len(oi2))                                            # 난수 하나로 사유를 구간별로 판정
reason = np.where(r_draw < p_opt_ret, "option_mismatch",
          np.where(r_draw < p_opt_ret + 0.02, rng.choice(["change_of_mind", "damaged", "wrong_delivery"], len(oi2), p=[.55, .30, .15]), None))
ret = oi2[reason != None].copy()                                         # 반품이 생긴 주문 상품만 남김
ret["return_reason"] = reason[reason != None]
ret["return_date"] = (ret["order_datetime"].dt.normalize() + pd.to_timedelta(rng.integers(2, 11, len(ret)), unit="D"))   # 주문 2~10일 뒤 접수
ret = ret[ret["return_date"] <= PRE_END]                                 # 데이터 추출 시점(8/17) 이후 접수 건은 아직 없으므로 제외
ret["quantity"] = 1
ret["refund_amount"] = ret["price"]
ret["return_id"] = [f"R{n:06d}" for n in range(1, len(ret) + 1)]
returns = ret[["return_id", "order_id", "item_id", "return_date", "quantity", "refund_amount", "return_reason"]].reset_index(drop=True)

# users: 기간 중 온라인 첫 구매가 생긴 회원은 첫 온라인 구매일 갱신 (데이터 추출 시점의 스냅샷)
# → 분석할 때 "기준 시점 이전 구매 이력"으로 고객을 나눠야 하는 이유
first_buy = orders.groupby("user_id")["order_datetime"].min().dt.normalize()
users["first_online_purchase_date"] = users["first_online_purchase_date"].fillna(users["user_id"].map(first_buy))
print("orders", len(orders), "order_items", len(order_items), "returns", len(returns), returns["return_reason"].value_counts().to_dict())


orders 51523 order_items 144768 returns 5370 {'option_mismatch': 2802, 'change_of_mind': 1421, 'damaged': 762, 'wrong_delivery': 385}


## 6라운드. 월별 KPI (kpi_monthly, 2025-01 ~ 2026-07)

- 경영 리포트용 전사 집계 (샘플 데이터와 직접 연결되지 않음)
- 2026-01 무료배송 기준 상향 이후 신규 첫 구매 전환율 계단식 하락, 2026-07 마케팅 +40%
- 2025-07 / 2026-07 값은 분석 요청서 지수(+12%)와 맞도록 고정

In [8]:
months = pd.date_range("2025-01-01", "2026-07-01", freq="MS")           # 2025-01 ~ 2026-07 매월 1일
SEASON = {1: .97, 2: .93, 3: 1.0, 4: 1.0, 5: 1.03, 6: 1.02, 7: 1.0, 8: 1.0, 9: 1.0, 10: 1.02, 11: 1.18, 12: 1.08}   # 월별 계절성 (11월 대형 세일)
kpi = []
for m in months:
    y, mo = m.year, m.month
    se = SEASON[mo]
    noise = lambda sd=0.02: 1 + rng.normal(0, sd)                         # 월별 자연스러운 흔들림 (기본 ±2%)
    if y == 2025:                                                         # 2025년: 무료배송 기준 2만 원 시절
        mkt, vis, sign, rate, avg1 = 30e8 * se * noise(), 700_000 * se * noise(), 150_000 * se * noise(), 0.17 + rng.normal(0, .003), 27_500
        rev_exist = 191.5e8 * se * noise(0.01)
    elif mo <= 6:                                                         # 2026년 1~6월: 기준 3만 원으로 상향 → 신규 전환율 계단식 하락
        mkt, vis, sign, rate, avg1 = 31e8 * se * noise(), 750_000 * se * noise(), 160_000 * se * noise(), 0.13 + rng.normal(0, .003), 29_000
        rev_exist = 191.5e8 * se * 1.135 * noise(0.01)                     # 기존 고객 매출은 전년 대비 +13.5% 수준 유지
    else:                                                                 # 2026년 7월: 마케팅 확대 (+40%) → 방문은 늘었지만 전환율은 더 낮음
        mkt, vis, sign, rate, avg1 = 42e8, 950_000, 220_000, 0.123, 29_000
        rev_exist = 218.0e8
    if (y, mo) == (2025, 7):                                              # 분석 요청서 지수(+12%)와 맞추기 위해 2025년 7월은 고정값
        mkt, vis, rate, rev_exist = 30e8, 700_000, 0.17, 191.5e8
    buyers = int(vis * rate)                                              # 신규 구매 고객 수 = 신규 방문 × 전환율
    avg_first = int(avg1 + rng.normal(0, 300))                            # 신규 첫 주문 평균 금액 (월별 흔들림 포함)
    kpi.append(dict(month=m.date(), new_visitors=int(vis), new_signups=int(sign), new_buyers=buyers,
                    new_first_purchase_rate=round(buyers / int(vis), 4), new_avg_first_order_amount=avg_first,
                    existing_buyers=int(rev_exist / 48_000), existing_purchase_rate=round(0.22 + rng.normal(0, .003), 4),
                    # 신규 매출 = 신규 구매 고객 수 × 첫 주문 평균 금액 (KPI 트리 곱셈 관계가 정확히 성립하도록 같은 값 사용)
                    revenue_new=buyers * avg_first, revenue_existing=int(rev_exist), marketing_cost=int(mkt)))
kpi_monthly = pd.DataFrame(kpi)
# 확인: 2026년 7월 온라인 매출이 전년 동월 대비 약 +12%인지
j25 = kpi_monthly.loc[kpi_monthly["month"] == pd.Timestamp("2025-07-01").date()].iloc[0]
j26 = kpi_monthly.loc[kpi_monthly["month"] == pd.Timestamp("2026-07-01").date()].iloc[0]
tot25, tot26 = j25.revenue_new + j25.revenue_existing, j26.revenue_new + j26.revenue_existing
print("7월 온라인 매출 YoY", round(tot26 / tot25 - 1, 3), "신규", round(j26.revenue_new / 1e8, 1), "억 / 전년", round(j25.revenue_new / 1e8, 1), "억")


7월 온라인 매출 YoY 0.121 신규 33.5 억 / 전년 32.8 억


## 7라운드. 정답 대조

- 세션 계획표로 계산한 값이 정답 문서 6-10 예상 결과와 맞는지 확인
- 맞지 않으면 생성 규칙이 의도대로 심어지지 않은 것

In [9]:
# 세션 계획표(정답)로 계산한 결과가 정답 문서 6-10 예상 결과와 맞는지 확인
cart = s[s["cart"]]                                                       # 장바구니 화면까지 간 세션
seg_order = ["online", "store", "new"]
check = pd.DataFrame({
    "장바구니→주문서": cart.groupby("segment")["checkout"].mean(),
    "부족 3천~8천": cart[cart["gap_bucket"] == 2].groupby("segment")["checkout"].mean(),
    "기준 이상": cart[cart["gap_bucket"] == 0].groupby("segment")["checkout"].mean(),   # 세 그룹이 비슷해야 정상 (원인은 고객 유형이 아니라 기준 미달)
    "8주 구매율": s.groupby(["segment", "u"])["purchase"].any().groupby("segment").mean(),
    "평균 결제 금액": s[s["purchase"]].groupby("segment")["final_amount"].mean(),
}).loc[seg_order]
rev = s[s["purchase"]].groupby("segment")["final_amount"].sum()
check["매출 비중"] = (rev / rev.sum()).loc[seg_order]
print(check.round(3))
print("buckets new:", cart[cart.segment=="new"].groupby("gap_bucket")["checkout"].mean().round(3).to_dict())   # 신규의 구간별 전환율 (U자형인지)
print("total", round(time.time() - T0, 1), "s")


         장바구니→주문서  부족 3천~8천  기준 이상  8주 구매율   평균 결제 금액  매출 비중
segment                                                     
online      0.669     0.511  0.723   0.519  45771.699  0.866
store       0.535     0.346  0.718   0.187  26302.051  0.038
new         0.517     0.332  0.717   0.247  23639.259  0.095
buckets new: {0: 0.717, 1: 0.448, 2: 0.332, 3: 0.449, 4: 0.548}
total 78.0 s


## 8라운드. 로컬 저장

- `data/`에 csv.gz로 저장 (events는 4라운드에서 이미 저장, 용량이 커서 GitHub에는 올리지 않음)
- 날짜 컬럼은 시각 없이 날짜만 남김

In [10]:
# 스키마(04 문서)에 있는 컬럼만 남김 (세그먼트 등 정답 정보는 제외)
products_out = products[["item_id", "item_name", "item_category", "item_variant", "option_count",
                         "is_mini", "price", "margin_rate"]]
users_out = users[["user_id", "signup_date", "signup_channel", "membership_grade",
                   "first_store_purchase_date", "first_online_purchase_date"]].copy()
for c in ["signup_date", "first_store_purchase_date", "first_online_purchase_date"]:
    users_out[c] = users_out[c].dt.date                      # 시각 없이 날짜만 남김 (BigQuery DATE)
returns_out = returns.copy()
returns_out["return_date"] = returns_out["return_date"].dt.date

TABLES = {"products": products_out, "users": users_out, "orders": orders,
          "order_items": order_items, "returns": returns_out, "kpi_monthly": kpi_monthly}
for name, df in TABLES.items():
    df.to_csv(f"{DATA_DIR}/{name}.csv.gz", index=False, compression="gzip")   # csv.gz로 압축 저장 (60일 만료·재적재 대비 백업)
    print(f"{name:12s} {len(df):>9,}행 저장")


products           587행 저장
users          100,000행 저장
orders          51,523행 저장
order_items    144,768행 저장
returns          5,370행 저장
kpi_monthly         19행 저장


## 9라운드. BigQuery 적재 (raw)

- 일반 테이블: DataFrame을 바로 적재 (`WRITE_TRUNCATE`라 다시 실행하면 덮어씀)
- events: 중첩 구조(event_params, items 등)라 **스키마를 직접 정의하고 JSON 파일을 업로드**
  - DataFrame으로 바로 올리면 중첩 구조 타입 추론이 불안정하고, 440만 행을 메모리에 올려야 해서 파일 업로드 방식을 선택
- 마지막에 테이블별 행 수를 조회해 로컬 결과와 비교

In [11]:
from google.cloud import bigquery           # BigQuery 클라이언트
import pydata_google_auth                   # 브라우저 로그인 인증

# 연결 테스트 노트북과 같은 방식으로 인증 (처음 한 번 로그인하면 이후에는 저장된 인증 정보 재사용)
credentials = pydata_google_auth.get_user_credentials(["https://www.googleapis.com/auth/bigquery"])
client = bigquery.Client(project=PROJECT_ID, credentials=credentials, location=LOCATION,          # location: 서울 리전에서 실행
                         default_query_job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES_BILLED))   # 쿼리 1회 최대 처리량 제한 (요금 안전장치)

# 날짜·일시 컬럼은 타입을 직접 지정
# (지정하지 않으면 order_datetime이 TIMESTAMP(UTC 기준 시각)로 올라가서 "KST 일시"라는 의미가 사라짐)
FIX_SCHEMA = {
    "users": [bigquery.SchemaField("signup_date", "DATE"),
              bigquery.SchemaField("first_store_purchase_date", "DATE"),
              bigquery.SchemaField("first_online_purchase_date", "DATE")],
    "orders": [bigquery.SchemaField("order_datetime", "DATETIME")],
    "returns": [bigquery.SchemaField("return_date", "DATE")],
    "kpi_monthly": [bigquery.SchemaField("month", "DATE")],
}
for name, df in TABLES.items():                                                  # 일반 테이블 6개
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE",      # 테이블이 있으면 비우고 다시 적재 (재실행해도 중복 없음)
                                        schema=FIX_SCHEMA.get(name, []))         # 지정한 컬럼만 타입 고정, 나머지는 자동 추론
    client.load_table_from_dataframe(df, f"{PROJECT_ID}.raw.{name}", job_config=job_config).result()   # .result(): 적재가 끝날 때까지 기다림
    print(f"raw.{name} 적재 완료")

# events: GA4 중첩 구조라 스키마를 직접 정의하고, 저장해둔 JSON 파일을 그대로 업로드
def rec(name, fields, mode="NULLABLE"):
    """중첩 필드(RECORD) 정의 도우미. mode="REPEATED"면 배열"""
    return bigquery.SchemaField(name, "RECORD", mode=mode, fields=fields)
VALUE = [bigquery.SchemaField("string_value", "STRING"), bigquery.SchemaField("int_value", "INTEGER"),
         bigquery.SchemaField("double_value", "FLOAT")]                          # GA4 value 구조: 타입별 칸 중 하나만 채워짐
KEY_VALUE = [bigquery.SchemaField("key", "STRING"), rec("value", VALUE)]         # event_params, user_properties의 원소 구조
EVENT_SCHEMA = [
    bigquery.SchemaField("event_date", "STRING"),                                # YYYYMMDD (KST)
    bigquery.SchemaField("event_timestamp", "INTEGER"),                          # 마이크로초 (UTC)
    bigquery.SchemaField("event_name", "STRING"),
    rec("event_params", KEY_VALUE, mode="REPEATED"),                              # 이벤트 속성 배열
    bigquery.SchemaField("user_id", "STRING"),
    bigquery.SchemaField("user_pseudo_id", "STRING"),
    bigquery.SchemaField("platform", "STRING"),
    rec("device", [bigquery.SchemaField("category", "STRING"), bigquery.SchemaField("operating_system", "STRING")]),
    rec("traffic_source", [bigquery.SchemaField("name", "STRING"), bigquery.SchemaField("medium", "STRING"),
                           bigquery.SchemaField("source", "STRING")]),
    rec("user_properties", KEY_VALUE, mode="REPEATED"),                           # 회원 속성 배열
    rec("items", [bigquery.SchemaField("item_id", "STRING"), bigquery.SchemaField("item_name", "STRING"),
                  bigquery.SchemaField("item_category", "STRING"), bigquery.SchemaField("item_variant", "STRING"),
                  bigquery.SchemaField("price", "FLOAT"), bigquery.SchemaField("quantity", "INTEGER")], mode="REPEATED"),   # 상품 배열
    rec("ecommerce", [bigquery.SchemaField("transaction_id", "STRING"), bigquery.SchemaField("purchase_revenue", "FLOAT"),
                      bigquery.SchemaField("shipping_value", "FLOAT"), bigquery.SchemaField("total_item_quantity", "INTEGER")]),   # 주문 금액 (purchase만)
]
job_config = bigquery.LoadJobConfig(source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,   # 줄마다 JSON 1개 형식
                                    schema=EVENT_SCHEMA, write_disposition="WRITE_TRUNCATE")
with open(f"{DATA_DIR}/events.jsonl.gz", "rb") as f:                            # gzip 압축 파일 그대로 업로드 (BigQuery가 압축을 풀어 읽음)
    client.load_table_from_file(f, f"{PROJECT_ID}.raw.events", job_config=job_config).result()
print("raw.events 적재 완료")

# 적재 확인: 테이블별 행 수 (로컬에서 만든 행 수와 같아야 함)
for name in list(TABLES) + ["events"]:
    cnt = client.query(f"SELECT COUNT(*) AS n FROM `{PROJECT_ID}.raw.{name}`").to_dataframe(create_bqstorage_client=False)["n"][0]
    print(f"{name:12s} {cnt:>10,}")


raw.products 적재 완료
raw.users 적재 완료
raw.orders 적재 완료
raw.order_items 적재 완료
raw.returns 적재 완료
raw.kpi_monthly 적재 완료
raw.events 적재 완료
products            587
users           100,000
orders           51,523
order_items     144,768
returns           5,370
kpi_monthly          19
events        4,390,472
